In [1]:
import math
from collections import Counter, defaultdict

class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus = [self.tokenize(doc) for doc in corpus]
        self.doc_lens = [len(doc) for doc in self.corpus]
        self.avgdl = sum(self.doc_lens) / len(self.doc_lens)
        self.df = self.compute_doc_freqs()
        self.idf = self.compute_idf()

    def tokenize(self, text):
        return text.lower().split()

    def compute_doc_freqs(self):
        """ Document frequency of each term """
        df = defaultdict(int)
        for doc in self.corpus:
            for term in set(doc):
                df[term] += 1
        return df

    def compute_idf(self):
        """ Compute IDF for each term """
        idf = {}
        N = len(self.corpus)
        for term, freq in self.df.items():
            idf[term] = math.log(1 + (N - freq + 0.5) / (freq + 0.5))
        return idf

    def score(self, query, index):
        """ Compute BM25 score for a single document """
        doc = self.corpus[index]
        doc_len = self.doc_lens[index]
        tf = Counter(doc)
        score = 0.0
        for term in self.tokenize(query):
            if term in tf:
                term_freq = tf[term]
                term_idf = self.idf.get(term, 0)
                denom = term_freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                score += term_idf * (term_freq * (self.k1 + 1)) / denom
        return score

    def rank(self, query, top_k=None):
        """ Rank all documents for a given query """
        scores = [(i, self.score(query, i)) for i in range(len(self.corpus))]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k] if top_k else scores


In [2]:
import json

# Load the JSON file
with open("eng_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Extract content (or title+summary if you prefer shorter index text)
documents = [article["title"] + article["content"]  for article in data if "content" in article]

# Initialize BM25 with extracted documents
bm25 = BM25(documents)

# Define a query
query = "student strike"

# Get top 5 results
top_docs = bm25.rank(query, top_k=5)

# Display ranked documents with their scores
for idx, score in top_docs:
    print(f"Score: {score:.4f}")
    print(f"Title: {data[idx].get('title')}")
    print(f"Link: {data[idx].get('link')}")
    print("-" * 60)

Score: 5.1065
Title: US Congressman Ro Khanna condemns Trump’s decision on Harvard international student ban
Link: https://nepalnews.com/s/international/us-congressman-ro-khanna-condemns-trumps-decision-on-harvard-international-student-ban/
------------------------------------------------------------
Score: 1.7174
Title: Gods for Men, Duties for Women
Link: https://nepalnews.com/s/opinion/gods-for-men-duties-for-women/
------------------------------------------------------------
Score: 0.9541
Title: Nepal News Evening Briefing | Friday, May 23, 2025
Link: https://nepalnews.com/s/main-left/nepal-news-evening-briefing-thursday-may-23-2025/
------------------------------------------------------------
Score: 0.0000
Title: PAC recommends CIAA probe into Patanjali land deal
Link: https://nepalnews.com/s/nation/pac-recommends-ciaa-probe-into-patanjali-land-deal/
------------------------------------------------------------
Score: 0.0000
Title: Deuba to hold discussions with top NC leaders
Link

In [4]:
%pip install transformers

  Using cached tokenizers-0.21.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.5.3-cp38-abi3-win_amd64.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   ---------------------------------------- 0.1/10.5 MB 3.2 MB/s eta 0:00:04
    --------------------------------------- 0.2/10.5 MB 3.1 MB/s eta 0:00:04
   -- ------------------------------------- 0.7/10.5 MB 5.6 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/10.5 MB 10.6 MB/s eta 0:00:01
   --------------- ------------------------ 4.0/10.5 MB 18.3 MB/s eta 0:00:01
   --------------------------- ------------ 7.1/10.5 MB 28.4 MB/s eta 0:00:01
   ---------------------------- ----------- 7.5/10.5 MB 25.3 MB/s eta 0:00:01
   ------------------------------ --------- 7.9/10.5 MB 24.1 MB/s eta 0:00:01
   --------------------------------- ------ 8.8/10.5 MB 21.8 MB/s eta 0:00:01
   ------------------------------------ --- 9.6/10.5 MB 22.0 MB/s eta 0:00:01
   


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from transformers import MBartForConditionalGeneration, MBart50Tokenizer

# ✅ Always load tokenizer from base model (not checkpoint)
tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

# ✅ Load model weights from your training checkpoint
model = MBartForConditionalGeneration.from_pretrained("./checkpoint-17000")

# Set language codes
tokenizer.src_lang = "ne_XX"
tokenizer.tgt_lang = "en_XX"

: 

In [14]:
!pip install sentencepiece


   ---------------------------------------- 0.0/992.0 kB ? eta -:--:--
   ------------------------------ -------- 786.4/992.0 kB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 992.0/992.0 kB 9.4 MB/s eta 0:00:00


In [10]:
%pip install --upgrade transformers sentencepiece


   ---------------------------------------- 0.0/991.5 kB ? eta -:--:--
    -------------------------------------- 20.5/991.5 kB 682.7 kB/s eta 0:00:02
   -- ------------------------------------ 71.7/991.5 kB 787.7 kB/s eta 0:00:02
   --------- ------------------------------ 225.3/991.5 kB 1.7 MB/s eta 0:00:01
   -------------------- ------------------- 501.8/991.5 kB 2.9 MB/s eta 0:00:01
   ---------------------------------------  983.0/991.5 kB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 991.5/991.5 kB 4.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
%pip list

Package                      VersionNote: you may need to restart the kernel to use updated packages.

---------------------------- ------------
absl-py                      2.0.0
asgiref                      3.7.2
asttokens                    2.4.1
astunparse                   1.6.3
attrs                        23.2.0
bleach                       6.1.0
blinker                      1.9.0
cachetools                   5.3.2
certifi                      2023.11.17
cffi                         1.16.0
charset-normalizer           3.3.2
click                        8.1.7
colorama                     0.4.6
comm                         0.2.2
comtypes                     1.4.5
contourpy                    1.2.1
cycler                       0.12.1
debugpy                      1.8.1
decorator                    5.1.1
Django                       5.0
executing                    2.0.1
filelock                     3.13.3
Flask                        3.1.0
flatbuffers                  23.5.26
fontto


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
